# EDA 01 — Pacientes y lista de espera

La **foto congelada**: quién espera, desde cuándo, por qué especialidad y con qué prioridad inicial.

Dataset sintético HLF · 500 pacientes · fecha de referencia 2026-10-13. Esquema en `docs/datos.md`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from carga import cargar, base_paciente, FECHA_REFERENCIA, ORDEN_PRIORIDAD, RANGOS_NORMALES

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 30)
COLOR_PRIO = {"alta": "#c0392b", "media": "#e67e22", "baja": "#2e86c1"}

d = cargar()
pac = base_paciente(d)
lista, eventos, mediciones, agenda, citas = d["lista"], d["eventos"], d["mediciones"], d["agenda"], d["citas"]

## 1. Calidad e integridad

In [ ]:
resumen = pd.DataFrame({
    nombre: {"filas": len(df), "columnas": df.shape[1], "nulos": int(df.isna().sum().sum()),
             "duplicados": int(df.duplicated().sum())}
    for nombre, df in d.items()
}).T
resumen

In [ ]:
# Nulos por columna (solo las que tienen)
pd.concat({n: df.isna().sum()[lambda s: s > 0] for n, df in d.items()}).rename("nulos").to_frame()

In [ ]:
checks = {
    "id_paciente único en pacientes": pac["id_paciente"].is_unique,
    "una solicitud por paciente": lista["id_paciente"].is_unique and len(lista) == len(d["pacientes"]),
    "fallecido ⇔ estado = fallecido": ((pac["fallecido"]) == (pac["estado"] == "fallecido")).all(),
    "fallecido ⇔ fecha_defuncion": ((pac["fallecido"]) == pac["fecha_defuncion"].notna()).all(),
    "defunción posterior al ingreso": (pac.dropna(subset=["fecha_defuncion"]).eval("fecha_defuncion >= fecha_ingreso_lista")).all(),
}
pd.Series(checks, name="ok").to_frame()

## 2. Demografía

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(pac, x="edad", hue="sexo", multiple="stack", bins=18, ax=ax[0]).set_title("Edad por sexo")
pac["comuna"].value_counts().plot.barh(ax=ax[1], color="#5d6d7e").set_title("Comuna")
ax[1].invert_yaxis()
pd.crosstab(pac["prevision"], pac["nivel_socioeconomico"]).plot.bar(stacked=True, ax=ax[2], rot=0).set_title("Previsión × nivel socioeconómico")
plt.tight_layout()

In [ ]:
pac[["edad"]].describe().T.join(pac["sexo"].value_counts(normalize=True).rename(lambda s: f"% {s}").to_frame().T.reset_index(drop=True).set_index(pd.Index(["edad"])))

## 3. Composición de la lista

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ct = pd.crosstab(lista["especialidad"], lista["prioridad_inicial"])
ct.loc[ct.sum(1).sort_values().index].plot.barh(stacked=True, ax=ax[0], color=[COLOR_PRIO[c] for c in ct.columns]).set_title("Solicitudes por especialidad y prioridad inicial")
pd.crosstab(lista["origen_derivacion"], lista["prioridad_inicial"], normalize="index").mul(100).plot.barh(
    stacked=True, ax=ax[1], color=[COLOR_PRIO[c] for c in ORDEN_PRIORIDAD]).set_title("% de prioridad inicial según origen de derivación")
ax[1].set_xlabel("%")
plt.tight_layout()

In [ ]:
(lista.groupby(["especialidad", "sub_especialidad", "diagnostico_cie10", "diagnostico_texto"])
      .agg(n=("id_solicitud", "size"),
           pct_alta=("prioridad_inicial", lambda s: round((s == "alta").mean() * 100, 1)))
      .sort_values("n", ascending=False))

In [ ]:
pd.crosstab(lista["estado"], lista["prioridad_inicial"], margins=True)

## 4. ¿Qué explica la prioridad inicial?

Si la prioridad inicial dependiera de la gravedad clínica, debería asociarse con el origen (urgencia / hospitalización) y quizá con la edad. Prueba χ² para variables categóricas.

In [ ]:
from scipy.stats import chi2_contingency, kruskal

filas = []
for var in ["especialidad", "origen_derivacion", "sexo", "comuna", "prevision", "nivel_socioeconomico"]:
    t = pd.crosstab(pac[var], pac["prioridad_inicial"])
    chi2, p, dof, _ = chi2_contingency(t)
    v = np.sqrt(chi2 / (t.values.sum() * (min(t.shape) - 1)))
    filas.append({"variable": var, "chi2": round(chi2, 1), "gl": dof, "p": round(p, 4), "V de Cramér": round(v, 3)})
h, p = kruskal(*[g["edad"] for _, g in pac.groupby("prioridad_inicial", observed=True)])
filas.append({"variable": "edad (Kruskal-Wallis)", "chi2": round(h, 1), "gl": 2, "p": round(p, 4), "V de Cramér": np.nan})
pd.DataFrame(filas).set_index("variable")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(pac, x="prioridad_inicial", y="edad", order=ORDEN_PRIORIDAD, palette=COLOR_PRIO, ax=ax[0]).set_title("Edad por prioridad inicial")
pd.crosstab(pac["nivel_socioeconomico"], pac["prioridad_inicial"], normalize="index").mul(100).plot.bar(
    stacked=True, rot=0, ax=ax[1], color=[COLOR_PRIO[c] for c in ORDEN_PRIORIDAD]).set_title("% prioridad por nivel socioeconómico")
plt.tight_layout()

## 5. Tiempo de espera

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ingresos = pd.crosstab(lista["fecha_ingreso_lista"].dt.strftime("%Y-%m"), lista["prioridad_inicial"])
ingresos.plot.bar(stacked=True, ax=ax[0], color=[COLOR_PRIO[c] for c in ingresos.columns], width=0.9, rot=60)
ax[0].set_title("Ingresos a la lista por mes")
activos = lista[lista["estado"] == "en_espera"]
sns.boxplot(activos, x="dias_espera", y="especialidad", hue="prioridad_inicial", hue_order=ORDEN_PRIORIDAD, palette=COLOR_PRIO, ax=ax[1])
ax[1].set_title("Días de espera (en_espera, al 2026-10-13)")
plt.tight_layout()

In [ ]:
activos.groupby("prioridad_inicial", observed=True)["dias_espera"].describe().round(0)

In [ ]:
# Días hasta la atención para los atendidos (primera cita con asistio)
primera_asist = citas[citas["estado_cita"] == "asistio"].groupby("id_paciente")["fecha_cita"].min()
at = lista[lista["estado"] == "atendido"].set_index("id_paciente").join(primera_asist)
at["dias_hasta_atencion"] = (at["fecha_cita"] - at["fecha_ingreso_lista"]).dt.days
at.groupby("prioridad_inicial", observed=True)["dias_hasta_atencion"].describe().round(0)

## 6. Fallecidos en espera

Control de exclusión: deben salir de la lista.

In [ ]:
fall = pac[pac["fallecido"]].copy()
fall["dias_ingreso_a_defuncion"] = (fall["fecha_defuncion"] - fall["fecha_ingreso_lista"]).dt.days
fall[["id_paciente", "edad", "especialidad", "diagnostico_cie10", "prioridad_inicial", "origen_derivacion",
      "fecha_ingreso_lista", "fecha_defuncion", "dias_ingreso_a_defuncion"]].sort_values("fecha_defuncion")

In [ ]:
pd.DataFrame({
    "fallecidos %": pac.groupby("prioridad_inicial", observed=True)["fallecido"].mean().mul(100).round(1),
    "n": pac.groupby("prioridad_inicial", observed=True).size(),
})

## Hallazgos

1. **Datos íntegros.** Sin duplicados; los únicos nulos son esperados (`fecha_defuncion` de los vivos, `minutos_atencion_real` de las citas sin asistencia). Fallecido, estado y fecha de defunción son coherentes.
2. **La prioridad inicial no se explica por ninguna variable disponible.** Ni especialidad, origen de derivación, edad, sexo, comuna, previsión ni nivel socioeconómico se asocian con ella (todas p > 0,2; V de Cramér ≤ 0,12). Incluso los derivados desde urgencia u hospitalización no tienen más prioridad alta. La foto inicial se comporta como ruido respecto de lo observable.
3. **La prioridad no ordena la atención.** Los pacientes en espera con prioridad alta llevan casi lo mismo que los de baja (mediana 301 vs. 335 días). Entre los atendidos, los de prioridad alta esperaron *más* (mediana 306 días) que los de media (210).
4. **Esperas largas.** La mediana de espera de los 409 pacientes activos es de unos 300 días.
5. **Fallecidos:** 15 (3 %), más frecuentes en prioridad alta (5,3 %) que en baja (1,8 %). Uno de ellos fallece 32 días después de ingresar (PAC-HLF-00009).
